#1. 연속형 vs 연속형 일떄는 피어슨 상관계수를 쓴다. 
# label이 명목형일때는 연관성을 어떻게 측정하는지 고찰 -> 스피어만 상관계수, 피어슨 상관계수가 뭔지 알아보자
#2. k-means를 통한 클러스터 파생변수 추가
#3. 파이캐럿 AutoML을 돌려서 상위 3개 모델을 선정
#4. catBoost를 블렌더 모델로 선정하여 전방 모델은 2번 상위모델 3개로 배치 = 스택킹
#5. 학습하여 test.csv를 찍어서 submission.csv를 제출
#6. 챗지피티 사용 가능하나, 랜덤뽑기로 설명하는 사람 선정

In [1]:
# pycaret 설치 확인 
from pycaret.classification import setup, compare_models
from sklearn.datasets import load_iris
import pandas as pd

data = load_iris(as_frame=True).frame
clf = setup(data=data, target='target', session_id=123)
best_model = compare_models()
print(best_model)

,Description,Value
0,Session id,123
1,Target,target
2,Target type,Multiclass
3,Original data shape,"(150, 5)"
4,Transformed data shape,"(150, 5)"
5,Transformed train set shape,"(105, 5)"
6,Transformed test set shape,"(45, 5)"
7,Numeric features,4
8,Preprocess,True
9,Imputation type,simple


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9718,0.0000,0.9718,0.9780,0.9712,0.9573,0.9609,0.2880
knn,K Neighbors Classifier,0.9718,0.9830,0.9718,0.9780,0.9712,0.9573,0.9609,0.0160
qda,Quadratic Discriminant Analysis,0.9718,0.0000,0.9718,0.9780,0.9712,0.9573,0.9609,0.0080
lda,Linear Discriminant Analysis,0.9718,0.0000,0.9718,0.9780,0.9712,0.9573,0.9609,0.0080
lightgbm,Light Gradient Boosting Machine,0.9536,0.9935,0.9536,0.9634,0.9528,0.9298,0.9356,10.3070
nb,Naive Bayes,0.9445,0.9868,0.9445,0.9525,0.9438,0.9161,0.9207,0.0090
et,Extra Trees Classifier,0.9445,0.9935,0.9445,0.9586,0.9426,0.9161,0.9246,0.0530
catboost,CatBoost Classifier,0.9445,0.9922,0.9445,0.9586,0.9426,0.9161,0.9246,0.4670
gbc,Gradient Boosting Classifier,0.9355,0.0000,0.9355,0.9416,0.9325,0.9023,0.9083,0.0610
dt,Decision Tree Classifier,0.9264,0.9429,0.9264,0.9502,0.9201,0.8886,0.9040,0.0090


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=123, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)


In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import f1_score
import catboost as ctb
import lightgbm as lgb
import os
import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
# 데이터 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
# 상관 관계확인 ID & 나머지 columns / y = support_needs

In [4]:
train_df.head(5), test_df.head(5)

(            ID   age gender  tenure  frequent  payment_interval  \
 0  TRAIN_00000  54.0      F    47.0      22.0               8.0   
 1  TRAIN_00001  30.0      M    16.0      15.0               5.0   
 2  TRAIN_00002  29.0      M     8.0      30.0              21.0   
 3  TRAIN_00003  38.0      F    38.0      23.0              10.0   
 4  TRAIN_00004  25.0      F    52.0       3.0              17.0   
 
   subscription_type  contract_length  after_interaction  support_needs  
 0            member               90               25.0              0  
 1               vip              360               23.0              0  
 2              plus               30               21.0              0  
 3               vip               90                6.0              0  
 4            member               30                1.0              2  ,
            ID   age gender  tenure  frequent  payment_interval  \
 0  TEST_00000  18.0      M    40.0       6.0              15.0   
 1  TEST_00

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30858 entries, 0 to 30857
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 30858 non-null  object 
 1   age                30858 non-null  float64
 2   gender             30858 non-null  object 
 3   tenure             30858 non-null  float64
 4   frequent           30858 non-null  float64
 5   payment_interval   30858 non-null  float64
 6   subscription_type  30858 non-null  object 
 7   contract_length    30858 non-null  int64  
 8   after_interaction  30858 non-null  float64
 9   support_needs      30858 non-null  int64  
dtypes: float64(5), int64(2), object(3)
memory usage: 2.4+ MB


In [ ]:
train_df.columns
# ['ID', 'age', 'gender', 'tenure', 'frequent', 'payment_interval', 'subscription_type', 'contract_length', 'after_interaction']
# ['support_needs']

Index(['ID', 'age', 'gender', 'tenure', 'frequent', 'payment_interval',
       'subscription_type', 'contract_length', 'after_interaction',
       'support_needs'],
      dtype='object')

In [9]:
import scipy.stats as spearsonr

In [ ]:
X = train_df.drop(columns=['ID', 'support_needs'])
y = train_df['support_needs']

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 


In [14]:
X_train.head()

,age,gender,tenure,frequent,payment_interval,subscription_type,contract_length,after_interaction
11378,19.0,M,27.0,29.0,2.0,plus,360,4.0
18706,19.0,F,57.0,18.0,18.0,member,90,17.0
23515,42.0,M,45.0,28.0,0.0,vip,90,24.0
11239,41.0,M,45.0,12.0,16.0,vip,360,5.0
15659,20.0,M,18.0,24.0,3.0,member,90,16.0


In [15]:
y_train.head()

11378    0
18706    0
23515    2
11239    2
15659    0
Name: support_needs, dtype: int64

In [19]:
X_valid.head(), y_valid.head()

(        age gender  tenure  frequent  payment_interval subscription_type  \
 10527  56.0      M    56.0      16.0               6.0            member   
 15103  62.0      F    10.0      20.0              16.0            member   
 2626   41.0      M    57.0      21.0               7.0            member   
 28108  48.0      F    37.0       8.0               9.0              plus   
 885    46.0      M    47.0      30.0               1.0               vip   
 
        contract_length  after_interaction  
 10527               30               10.0  
 15103              360                7.0  
 2626                90               23.0  
 28108               90                5.0  
 885                360                1.0  ,
 10527    1
 15103    1
 2626     0
 28108    1
 885      2
 Name: support_needs, dtype: int64)

In [31]:
from pycaret.classification import setup, compare_models

train_df = pd.concat([X_train, y_train], axis=1)


In [40]:
from sklearn.cluster import KMeans
X_train_kmeans = X_train.drop(columns=['gender', 'subscription_type'])
X_valid_kmeans = X_valid.drop(columns=['gender', 'subscription_type'])

kmeans = KMeans(n_clusters=3, random_state=42)
X_train['cluster'] = kmeans.fit_predict(X_train_kmeans)
X_valid['cluster'] = kmeans.predict(X_valid_kmeans)


In [41]:
from pycaret.classification import setup, compare_models, blend_models, finalize_model, predict_model

clf_setup = setup(data=train_df, target='support_needs', session_id=42, normalize=True)

,Description,Value
0,Session id,42
1,Target,support_needs
2,Target type,Multiclass
3,Original data shape,"(24686, 9)"
4,Transformed data shape,"(24686, 11)"
5,Transformed train set shape,"(17280, 11)"
6,Transformed test set shape,"(7406, 11)"
7,Numeric features,6
8,Categorical features,2
9,Preprocess,True


In [42]:
train_df = pd.concat([X_train, y_train], axis=1)

In [44]:
top3_models = compare_models(n_select=3, exclude=['lightgbm'])

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,10:31:30
Status,. . . . . . . . . . . . . . . . . .,Loading Estimator
Estimator,. . . . . . . . . . . . . . . . . .,Logistic Regression


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.5347,0.0000,0.5347,0.4769,0.4666,0.2430,0.2641,0.8720
catboost,CatBoost Classifier,0.5081,0.6669,0.5081,0.4746,0.4780,0.2056,0.2123,5.2590
rf,Random Forest Classifier,0.5078,0.6620,0.5078,0.4770,0.4801,0.2090,0.2150,0.3680
ada,Ada Boost Classifier,0.5045,0.0000,0.5045,0.4447,0.4331,0.1560,0.1805,0.1240
nb,Naive Bayes,0.5041,0.6464,0.5041,0.4464,0.4155,0.1484,0.1799,0.0320
xgboost,Extreme Gradient Boosting,0.5031,0.6623,0.5031,0.4765,0.4814,0.2005,0.2048,0.2040
lda,Linear Discriminant Analysis,0.4938,0.0000,0.4938,0.4258,0.4117,0.1285,0.1554,0.0330
ridge,Ridge Classifier,0.4936,0.0000,0.4936,0.4210,0.4052,0.1239,0.1536,0.0340
lr,Logistic Regression,0.4936,0.0000,0.4936,0.4270,0.4122,0.1280,0.1547,0.3120
et,Extra Trees Classifier,0.4936,0.6482,0.4936,0.4616,0.4677,0.1802,0.1852,0.3470


In [47]:
from pycaret.classification import blend_models, create_model

catboost_final = create_model('catboost')
blender = blend_models(estimator_list=top3_models, choose_better=True, method='soft', fold=5)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4994,0.6561,0.4994,0.4689,0.4711,0.1891,0.1953
1,0.5220,0.6833,0.5220,0.4874,0.4917,0.2248,0.2323
2,0.5087,0.6722,0.5087,0.4697,0.4755,0.2086,0.2158
3,0.5197,0.6670,0.5197,0.4939,0.4922,0.2289,0.2358
4,0.4954,0.6600,0.4954,0.4541,0.4633,0.1834,0.1894
5,0.5260,0.6866,0.5260,0.4974,0.4957,0.2333,0.2415
6,0.4994,0.6641,0.4994,0.4625,0.4706,0.1934,0.1989
7,0.5046,0.6671,0.5046,0.4698,0.4733,0.1997,0.2066
8,0.4994,0.6524,0.4994,0.4653,0.4687,0.1910,0.1976


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.5295,0.6800,0.5295,0.4860,0.4850,0.2356,0.2481
1,0.5231,0.6767,0.5231,0.4776,0.4769,0.2326,0.2450
2,0.5286,0.6803,0.5286,0.4912,0.4859,0.2368,0.2489
3,0.5214,0.6775,0.5214,0.4802,0.4782,0.2259,0.2373
4,0.5142,0.6646,0.5142,0.4758,0.4712,0.2155,0.2267
Mean,0.5234,0.6758,0.5234,0.4822,0.4794,0.2293,0.2412
Std,0.0055,0.0058,0.0055,0.0057,0.0055,0.0079,0.0083


Original model was better than the blended model, hence it will be returned. NOTE: The display metrics are for the blended model (not the original one).


In [50]:
# 최종 모델 선정 = catBoost를 블렌더 모델
preds = predict_model(catboost_final, data=test_df)
preds[['ID', 'Label']].to_csv('submission.csv', index=False)

KeyError: "['Label'] not in index"